# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR^2 mass spectrometry dataset of extracellular vesicles from human mast cells using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the Croissant Dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset: {metadata.name}\n{metadata.description}")

## 2. Data Overview
List available record sets, with their `@id`, name, and fields.

In [ ]:
# List record sets with their @id, name, and contained fields
record_sets = list(dataset.record_sets())
print(f"Number of record sets: {len(record_sets)}\n")
record_set_ids = []
for rs in record_sets:
    print(f"Record Set @id: {rs.id}")
    print(f"  Name: {rs.name}")
    print(f"  Description: {rs.description}")
    print(f"  Fields/Columns:")
    for field in rs.fields:
        print(f"    - Field @id: {field.id}, name: {field.name}, dtype: {field.data_type}")
    print()
    record_set_ids.append(rs.id)

## 3. Data Extraction
Load data from one or more record sets into DataFrames. Record sets and field `@id`s are referenced explicitly.

In [ ]:
# Extract data from all record sets into DataFrames
dataframes = {}
for record_set_id in record_set_ids:
    print(f"\nLoading records from record set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"  Columns: {df.columns.tolist()}")
    print(f"  Sample:")
    display(df.head(2))

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps (e.g., filtering, normalization, grouping). Choose a numeric field and a group field by `@id` for analysis.

In [ ]:
# Example: Use the first record set (update as appropriate for analysis)
selected_record_set_id = record_set_ids[0]
df = dataframes[selected_record_set_id]

# Display the schema for this record set to help pick fields
rs_obj = next(rs for rs in record_sets if rs.id == selected_record_set_id)
numeric_fields = [field.id for field in rs_obj.fields if field.data_type in ("Number", "Float", "Integer")]
print(f"Numeric fields for {selected_record_set_id}: {numeric_fields}")
if not numeric_fields:
    print('No numeric fields found.')
else:
    numeric_field_id = numeric_fields[0]
    print(f"Using numeric field: {numeric_field_id}")

    # Filter for values above a threshold (if possible)
    if numeric_field_id in df:
        threshold = df[numeric_field_id].mean() if df[numeric_field_id].dtype in ['float64','int64'] else 0
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
        display(filtered_df.head())

        # Normalize
        filtered_df[f"{numeric_field_id}_normalized"] = (
            (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        )
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Try grouping by a categorical field
        group_fields = [field.id for field in rs_obj.fields if field.data_type in ("Text", "String") and field.id != numeric_field_id]
        group_field = group_fields[0] if group_fields else None
        if group_field and group_field in filtered_df.columns:
            grouped = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
            print(f"Grouped data by {group_field} (mean {numeric_field_id}):")
            display(grouped.head())
        else:
            print("No suitable group field found for this record set.")
    else:
        print(f"Field {numeric_field_id} not in DataFrame columns: {df.columns}")

## 5. Visualization
Visualize numeric field distributions and relationships, referencing fields by their `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if 'filtered_df' in locals() and not filtered_df.empty and numeric_field_id in filtered_df:
    plt.figure(figsize=(8, 5))
    sns.histplot(filtered_df[numeric_field_id], bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id} in filtered records")
    plt.xlabel(numeric_field_id)
    plt.show()
    
    # If group_field exists, boxplot
    if 'group_field' in locals() and group_field and group_field in filtered_df.columns:
        plt.figure(figsize=(10, 6))
        sns.boxplot(data=filtered_df, x=group_field, y=numeric_field_id)
        plt.title(f"{numeric_field_id} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
In this notebook, we've explored the FAIR^2 mass spectrometry dataset using the `mlcroissant` library. We reviewed dataset metadata, listed record sets and fields, extracted data, performed simple analysis (filtering, normalization, grouping), and visualized key quantitative variables. These steps provide a foundation for deeper proteomics or machine learning studies leveraging well-annotated Croissant datasets.